# 📝 과제 02: 온라인 쇼핑 상담 QLoRA 학습과 평가

## 만들 결과

교안 02(QLoRA 학습과 평가)의 과제입니다. 온라인 쇼핑 상담(전자상거래법) 데이터로 4비트 Base에 지난 과제와 같은 어댑터를 4에폭 학습하고, 에폭별 어댑터를 같은 판사 지표로 평가해 QLoRA Tuned를 고른 뒤 지난 과제의 LoRA Tuned와 비교하고 float32 베이스에 병합합니다.

- 1~4번: 학습 전·후 답 테이블 `answer_table`, 에폭마다 저장한 어댑터 `output/commerce_qlora_lr2e-4/checkpoint-<갱신 횟수>`, 학습 최대 할당 메모리 `max_allocated_gb`
- 5~6번: 지표 테이블 `forgetting_table`, 비교 테이블 `qlora_compare_table`
- 7번: 병합 모델 `merged_model`과 그 답 `merged_answers`

## 준비

- **API 키:** 판사는 지난 과제와 같은 Jev라 `.env`의 `TYPESAFE_API_KEY`가 필요합니다.
- **자료:** `data/과제/`의 `commerce_train.jsonl`, `commerce_val.jsonl`, `commerce_eval.jsonl`(평가셋), `commerce_out_of_scope_eval.jsonl`과 `data/본실습/`의 일반 질문, 일반 문단. 모두 지난 과제와 같은 파일입니다.
- **제공 코드:** 지난 과제와 같은 준비 셀(`generate_replies`, 데이터, MLflow, 판사와 `evaluate_model`, `table_metrics`, `METRIC_NAMES`)과 4비트 Base `base_4bit`입니다. 2번 뒤 LoRA 설정, 4번 뒤 4비트 Base 다시 불러오기, 5번 뒤 QLoRA Tuned 고르기(`tuned_row`, `tuned_name`)도 제공 셀입니다.
- **실행 환경:** 교안과 같은 RunPod CUDA GPU에서 실행합니다.
- **작성:** `[작성]` 셀의 `...`를 `# TODO` 주석대로 채우고 바로 아래 `[자가채점]` 셀을 실행합니다. 뒤 문항이 앞 문항의 변수를 쓰므로 1번부터 차례로 풉니다. 판사 결과와 측정값은 실행마다 달라지므로 자가채점은 테이블의 모양과 크기 순서를 봅니다.

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

# 토크나이저: 학습 전후 모델이 같은 토크나이저를 씁니다
# HF_TOKEN 이 없다는 Warning 은 로그인 없이 내려받는다는 안내이며 오류가 아닙니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("대화 끝 토큰:", tokenizer.eos_token, "| 패딩 토큰:", tokenizer.pad_token)

In [ ]:
# 지난 시간 과제 데이터의 시스템 프롬프트입니다(학습 데이터의 messages와 글자까지 같습니다)
SYSTEM_PROMPT_COMMERCE = "너는 온라인 쇼핑 상담 챗봇이다. 전자상거래법 조문에 근거해 존댓말로 간결히 답한다."

# 대화 끝 토큰 <|im_end|>의 id. 생성이 이 토큰에서 멈추도록 eos_token_id로 넘깁니다
END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")


def generate_replies(model, questions, system_prompt, max_new_tokens=256):
    """질문을 32개씩 묶어 생성하고, 질문 순서대로 답 목록을 돌려줍니다. system_prompt=None 이면 시스템 메시지 없이 묻습니다."""
    # 여러 질문을 묶으면 짧은 질문을 패딩으로 채워 길이를 맞춥니다. 답은 오른쪽 끝에 이어 쓰므로 패딩은 왼쪽에 둡니다
    tokenizer.padding_side = "left"
    answers = []
    for start in range(0, len(questions), 32):
        # generate_reply 와 같은 chat template으로 질문을 감쌉니다
        prompts = []
        for question in questions[start:start + 32]:
            messages = []
            if system_prompt is not None:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": question})
            prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False))
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

        # generate_reply 와 같은 그리디 생성
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                        eos_token_id=END_OF_TURN_ID, pad_token_id=tokenizer.pad_token_id)

        # 입력 길이 뒤의 새 토큰만 글자로 바꿉니다(먼저 끝난 답 뒤의 패딩은 skip_special_tokens 로 빠집니다)
        for new_ids in output_ids[:, inputs["input_ids"].shape[1]:]:
            answers.append(tokenizer.decode(new_ids, skip_special_tokens=True))

    tokenizer.padding_side = "right"  # 학습기가 쓰는 오른쪽 패딩으로 되돌립니다
    return answers


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def to_prompt_completion(row):
    """messages를 앞부분(system이 있으면 system, user)과 배울 답(마지막 assistant)으로 나눕니다."""
    prompt = row["messages"][:-1]      # 마지막 답 앞의 메시지(system, user)
    completion = row["messages"][-1:]  # 마지막 assistant 메시지 하나
    return {"prompt": prompt, "completion": completion}

In [ ]:
import mlflow

# 실험 기록은 output/mlflow.db 한 파일에 남깁니다(별도 서버 없이 sqlite)
mlflow.set_tracking_uri(f"sqlite:///{output_dir / 'mlflow.db'}")
# 처음 실행하면 실험을 새로 만든다는 INFO 안내(Creating a new experiment)가 나옵니다. 오류가 아닙니다
mlflow.set_experiment("commerce-qlora")
print("MLflow 기록 위치:", mlflow.get_tracking_uri())

In [ ]:
# 학습, 검증 데이터와 평가에 쓰는 네 가지 자료를 읽습니다
train_rows = load_jsonl(data_dir / "과제" / "commerce_train.jsonl")
val_rows = load_jsonl(data_dir / "과제" / "commerce_val.jsonl")
eval_rows = load_jsonl(data_dir / "과제" / "commerce_eval.jsonl")                       # 쇼핑 상담 평가셋: question_type 은 재현, 적용
out_of_scope_rows = load_jsonl(data_dir / "과제" / "commerce_out_of_scope_eval.jsonl")  # 범위 밖 질문
general_rows = load_jsonl(data_dir / "본실습" / "general_questions.jsonl")              # 일반 질문과 정답 답변(교안 02와 같은 파일)
general_texts = json.loads((data_dir / "본실습" / "general_texts.json").read_text(encoding="utf-8"))  # 일반 문단(perplexity)

print("학습:", len(train_rows), "| 검증:", len(val_rows), "| 평가셋:", len(eval_rows), "| 범위 밖:", len(out_of_scope_rows),
      "| 일반 질문:", len(general_rows), "| 일반 문단:", len(general_texts))
print("평가셋 종류별:", pd.DataFrame(eval_rows)["question_type"].value_counts().to_dict())

In [ ]:
from datasets import Dataset

# 학습기에 넣을 prompt, completion Dataset(지난 과제와 같은 형식)
train_dataset = Dataset.from_list([to_prompt_completion(row) for row in train_rows])
val_dataset = Dataset.from_list([to_prompt_completion(row) for row in val_rows])

# 학습 데이터 표본: 거절 답 행을 뺀 학습 행에서 같은 간격으로 고른 40행(지난 과제 02와 같은 행)
answer_rows = []
for row in train_rows:
    if "답할 수 없" not in row["answer"]:
        answer_rows.append(row)
sample_gap = len(answer_rows) // 40
train_sample_rows = []
for row in answer_rows[::sample_gap][:40]:
    # 학습한 답(answer)을 정답 답변으로 씁니다
    train_sample_rows.append({"question": row["question"], "reference_answer": row["answer"]})

아래 셀은 지난 과제 02와 같은 판사, 채점 함수, 지표 함수입니다. 실행만 하고 넘어갑니다.

In [ ]:
import os

from dotenv import load_dotenv

# 단원 폴더의 .env를 먼저 읽고, 없는 키만 상위 실습자료 폴더의 .env에서 채웁니다
load_dotenv(material_dir / ".env")
load_dotenv(material_dir.resolve().parent / ".env")

os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"  # 사용 기록을 외부로 보내지 않음(deepeval을 불러오기 전에 설정)
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig
from deepeval.metrics import JevEval
from deepeval.metrics.jev_eval import Noul
from deepeval.test_case import LLMTestCase, SingleTurnParams

# 판정 결과가 모델 버전에 따라 달라지지 않게 Jev 버전을 고정합니다
jev_model_name = os.getenv("TYPESAFE_MODEL", "jev-1.13.0")
print("Jev 모델:", jev_model_name)
print("Jev 키 읽음:", os.getenv("TYPESAFE_API_KEY") is not None)

In [ ]:
# 두 판사가 함께 쓰는 판정 문장. Jev가 문장마다 참일 확률을 돌려줍니다
BASE_STATEMENTS = [
    Noul("actual_output은 input의 질문에 직접 답한다."),
    Noul("actual_output의 사실(숫자, 기간, 조건, 이름)과 결론이 expected_output과 일치한다. 표현이 달라도 같은 정보를 전하면 일치로 본다."),
    Noul("actual_output은 input의 질문과 관계없는 지어낸 내용을 덧붙이지 않는다."),
    Noul("actual_output은 같은 문장이나 구절을 되풀이하지 않는다."),
]

# 일반 질문과 학습 데이터 표본용: 질문, 모델의 답, 정답 답변을 봅니다
general_metric = JevEval(
    name="정확성",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],  # 판정 문장에서 가리키는 필드
    questions=BASE_STATEMENTS,
    system_one_model=jev_model_name,
    strict_mode=True,  # 모든 문장의 참일 확률이 0.5 이상이어야 통과
    async_mode=True,  # Jev 호출을 비동기로 보냄(DeepEval 기본값)
)

# 쇼핑 상담 질문용: 근거 조문(context)도 보고, 정답 답변에 없는 말이 조문과 어긋나지 않는지 한 문장 더 묻습니다
law_metric = JevEval(
    name="정확성(근거 조문)",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT, SingleTurnParams.CONTEXT],
    questions=BASE_STATEMENTS + [
        Noul("actual_output이 expected_output에 없는 내용을 말할 때, 그 내용은 context(근거 조문)와 모순되지 않는다."),
    ],
    system_one_model=jev_model_name,
    strict_mode=True,
    async_mode=True,
)


def judge_answers(rows, answers):
    """판사가 답 목록을 동시에 채점해, 문항 순서대로 점수, 통과, 이유 열을 가진 테이블을 돌려줍니다."""
    # rows 는 모두 같은 종류입니다. 근거 조문(law_text)이 있으면 근거 조문 기준으로 채점합니다
    metric = general_metric
    if "law_text" in rows[0]:
        metric = law_metric

    # 문항마다 테스트 케이스 하나
    test_cases = []
    for index, row in enumerate(rows):
        context = None
        if "law_text" in row:
            context = [row["law_text"]]
        test_cases.append(LLMTestCase(
            name=str(index),                          # 결과를 문항 순서로 되돌리는 번호(동시 채점은 끝나는 순서가 섞입니다)
            input=row["question"],                    # 질문
            actual_output=answers[index],             # 채점할 모델의 답
            expected_output=row["reference_answer"],  # 정답 답변
            context=context,                          # 근거 조문 원문(없으면 None)
        ))

    # 판사를 동시에 부릅니다. 동시에 보낼 문항 수 상한은 64이고, 반 전체가 같은 키를 쓰다 429(요청 한도 초과)가 나면 줄입니다
    result = evaluate(test_cases=test_cases, metrics=[metric], async_config=AsyncConfig(max_concurrent=64),
                      display_config=DisplayConfig(print_results=False, show_indicator=False))  # 긴 기본 출력을 끕니다

    # 결과를 번호(name)로 찾아 문항 순서대로 놓습니다. 점수는 모든 문장이 참이면 1, 아니면 0이고, 이유에는 문장마다 참일 확률이 나옵니다
    judged = {}
    for test_result in result.test_results:
        judged[test_result.name] = test_result.metrics_data[0]  # 지표를 하나만 넘겼습니다
    judged_rows = []
    for index in range(len(rows)):
        metric_data = judged[str(index)]
        judged_rows.append({"점수": metric_data.score, "통과": metric_data.success, "이유": metric_data.reason})
    return pd.DataFrame(judged_rows)

In [ ]:
def answer_and_judge(model, rows, system_prompt):
    """rows의 질문에 답을 생성하고 채점해, 질문마다 한 행씩 모은 테이블을 돌려줍니다."""
    questions = []
    for row in rows:
        questions.append(row["question"])
    answers = generate_replies(model, questions, system_prompt)

    # 거절은 거절 키워드가 답에 있는지로 셉니다(거절률과 과잉 거절률에 씁니다)
    result_rows = []
    for row, answer in zip(rows, answers):
        result_rows.append({"질문": row["question"], "종류": row.get("question_type", ""), "답변": answer,
                            "거절": "답할 수 없" in answer})
    result_table = pd.DataFrame(result_rows)

    # 정답 답변이 있는 행만 판사가 채점합니다(범위 밖 질문은 정답 답변이 없습니다)
    if "reference_answer" in rows[0]:
        result_table = pd.concat([result_table, judge_answers(rows, answers)], axis=1)
    return result_table

In [ ]:
import math


def perplexity(model, texts):
    """문단 목록 전체의 perplexity: 예측한 토큰 전체의 평균 손실에 exp를 합니다."""
    total_loss = 0.0
    total_tokens = 0
    for text in texts:
        input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
        with torch.no_grad():
            loss = model(input_ids, labels=input_ids).loss.item()  # 입력을 정답으로 주면 다음 토큰 예측 손실의 평균을 돌려줍니다

        # 첫 토큰은 예측하지 않으므로 예측한 토큰 수는 길이 - 1입니다. 평균 * 개수 = 이 문단의 손실 합
        n_predicted = input_ids.shape[1] - 1
        total_loss = total_loss + loss * n_predicted
        total_tokens = total_tokens + n_predicted
    return math.exp(total_loss / total_tokens)


# 지표 이름(MLflow 에 기록할 이름)과 테이블에 보일 이름. 학습 데이터 정답률을 맨 앞에 둡니다
METRIC_NAMES = {"train_acc": "학습 데이터 정답률", "domain_recall_acc": "도메인 정답률(재현)", "domain_apply_acc": "도메인 정답률(적용)",
                "over_refusal_rate": "과잉 거절률", "refusal_rate": "거절률", "general_acc": "일반 질문 정답률", "general_ppl": "일반 perplexity"}


def evaluate_model(model):
    """모델 하나를 앞의 평가 네 가지와 perplexity 로 평가해 사전으로 돌려줍니다."""
    return {
        "train": answer_and_judge(model, train_sample_rows, SYSTEM_PROMPT_COMMERCE),
        "domain": answer_and_judge(model, eval_rows, SYSTEM_PROMPT_COMMERCE),
        "refusal": answer_and_judge(model, out_of_scope_rows, SYSTEM_PROMPT_COMMERCE),
        "general": answer_and_judge(model, general_rows, None),
        "ppl": perplexity(model, general_texts),
    }


def table_metrics(tables):
    """evaluate_model 이 돌려준 사전으로 지표 일곱 개를 계산합니다. 정답률, 거절률은 True 를 1로 본 평균입니다."""
    domain_table = tables["domain"]
    return {
        "train_acc": tables["train"]["통과"].mean(),
        "domain_recall_acc": domain_table[domain_table["종류"] == "재현"]["통과"].mean(),
        "domain_apply_acc": domain_table[domain_table["종류"] == "적용"]["통과"].mean(),
        "over_refusal_rate": domain_table["거절"].mean(),
        "refusal_rate": tables["refusal"]["거절"].mean(),
        "general_acc": tables["general"]["통과"].mean(),
        "general_ppl": tables["ppl"],
    }

아래 셀은 과제 01과 같은 4비트 설정으로 4비트 Base를 불러옵니다.

In [ ]:
from transformers import BitsAndBytesConfig

# QLoRA 논문과 같은 4비트 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # 선형 층 가중치를 4비트로 불러옵니다
    bnb_4bit_quant_type="nf4",              # 기본값 "fp4" 대신 NF4로 양자화합니다
    bnb_4bit_use_double_quant=True,         # 블록마다 둔 absmax도 8비트로 한 번 더 양자화합니다
    bnb_4bit_compute_dtype=torch.bfloat16,  # 계산할 때 bf16으로 되돌립니다(기본값 float32는 느립니다)
)

In [ ]:
# 4비트 Base: 선형 층은 4비트, 나머지 층은 bf16
base_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,                      # 허브의 모델 이름
    quantization_config=bnb_config,  # 선형 층을 NF4 4비트로 불러오는 설정
    dtype=torch.bfloat16,            # 4비트로 바뀌지 않는 층(임베딩, 정규화)의 dtype
    device_map={"": 0},              # 모델 전체("")를 GPU 0번(첫 번째 GPU)에 둡니다
)

## 1. 학습 전 답: generate_replies

**목적:** 학습 전 4비트 Base가 쇼핑 상담 질문에 어떻게 답하는지 남겨, 학습 뒤 답과 비교합니다.

**요구사항**
- (1) 셀에 적어 둔 `check_rows` 네 행의 `"question"`을 같은 순서로 모은 리스트 **`check_questions`** 를 만듭니다.
- (2) `generate_replies`로 `base_4bit`가 `SYSTEM_PROMPT_COMMERCE`를 시스템 프롬프트로 `check_questions`에 한 답의 리스트 **`before_answers`** 를 만듭니다.
- (3) 열이 `"질문"`(`check_questions`), `"정답 답변"`(`check_rows`의 `"answer"`), `"4비트 Base 답변"`(`before_answers`)이고 이 순서인 `pd.DataFrame` **`answer_table`** 을 만듭니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 02 2-1 학습 전 답 셀과 같은 방법입니다.
교안 02 2-1 은 generate_replies 를 감싼 answer_check_questions 를 씁니다. generate_replies(model, questions, system_prompt) 는 교안 02 1-2 에 정의되어 있습니다.
```

</details>

In [ ]:
# [작성] 질문 리스트와 4비트 Base 의 답을 만들어 테이블로 만드세요.

# 학습 데이터에서 정답 답변이 거절 답인 첫 행(그대로 씁니다)
for row in train_rows:
    if "답할 수 없" in row["answer"]:
        refusal_row = row
        break

# 학습 전후에 똑같이 물을 질문 행 네 개: 학습 데이터 2, 검증 데이터 1, 범위 밖 질문 1(그대로 씁니다)
check_rows = [train_rows[0], train_rows[90], val_rows[0], refusal_row]

# TODO (1) for 문으로 check_rows 를 돌며 row["question"] 을 check_questions 에 더합니다
check_questions = []
...

# 첫 생성에서 빠른 커널 대신 PyTorch 기본 구현으로 계산한다는 안내(causal_conv1d, flash-linear-attention 이 없다)가 나옵니다. 오류가 아닙니다
# TODO (2) generate_replies 에 base_4bit, check_questions, SYSTEM_PROMPT_COMMERCE 를 넘깁니다
before_answers = ...

# TODO (3) 열 이름을 키, 리스트를 값으로 한 딕셔너리를 pd.DataFrame 에 넘깁니다(정답 답변은 check_rows 의 "answer")
answer_table = ...
display(answer_table)

In [ ]:
# [자가채점] 질문 리스트, 답 리스트, 테이블의 열과 행을 확인합니다
expected_questions = [row["question"] for row in check_rows]
expected_references = [row["answer"] for row in check_rows]
assert check_questions == expected_questions, "check_questions는 check_rows 네 행의 question 을 순서 그대로 담습니다."
assert isinstance(before_answers, list) and len(before_answers) == 4, "before_answers는 질문 네 개에 대한 답 리스트입니다."
assert isinstance(answer_table, pd.DataFrame) and list(answer_table.columns) == ["질문", "정답 답변", "4비트 Base 답변"], f"answer_table의 열이 {list(answer_table.columns)}입니다. 지문의 열 이름과 순서를 확인하세요."
assert list(answer_table["질문"]) == expected_questions and list(answer_table["정답 답변"]) == expected_references, "\"질문\", \"정답 답변\" 열은 check_rows 의 question, answer 를 순서 그대로 담습니다."
assert list(answer_table["4비트 Base 답변"]) == before_answers, "\"4비트 Base 답변\" 열은 before_answers 입니다."
for answer in before_answers:
    assert isinstance(answer, str) and answer.strip() != "", "4비트 Base 답변이 비어 있습니다. generate_replies 의 반환값을 담았는지 확인하세요."
print("✅ 통과!")

## 2. 4비트 Base 지표: evaluate_model, table_metrics

**목적:** 학습 전 4비트 Base의 점수를 남겨, 학습한 어댑터와 비교할 기준을 만듭니다.

**요구사항**
- (1) `evaluate_model`로 `base_4bit`를 평가한 반환값을 **`base_tables`** 에 담습니다(3번에서 어댑터를 붙이기 전에 평가합니다).
- (2) 키 `"4비트 Base"`에 `table_metrics(base_tables)`의 지표를 담은 딕셔너리 **`model_metrics`** 를 만듭니다.
- (3) `model_metrics`의 모델이 행, 지표가 열이고 열 이름을 `METRIC_NAMES`의 테이블 이름으로 바꾼 `pd.DataFrame` **`forgetting_table`** 을 만듭니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 02 3-1 4비트 Base 지표 셀, 3-3 에폭별 어댑터 평가 셀과 같은 방법입니다.
```

</details>

In [ ]:
# [작성] 4비트 Base 를 평가하고 지표 딕셔너리와 테이블을 만드세요.

# 판사 호출마다 DeepEval 이 Pass Rate 요약, 'No hyperparameters logged' 경고, Confident AI 안내를 출력합니다. 오류가 아닙니다
# TODO (1) evaluate_model 로 base_4bit 를 평가합니다
base_tables = ...

# TODO (2) "4비트 Base" 키에 table_metrics 로 계산한 지표를 담습니다
model_metrics = ...

# TODO (3) model_metrics 로 모델이 행, 지표가 열이고 열 이름이 METRIC_NAMES 의 테이블 이름인 테이블을 만듭니다
forgetting_table = ...
display(forgetting_table.round(3))

In [ ]:
# [자가채점] 채점 딕셔너리의 키, 지표 딕셔너리, 테이블 모양을 확인합니다
assert isinstance(base_tables, dict) and list(base_tables) == ["train", "domain", "refusal", "general", "ppl"], "base_tables는 evaluate_model(base_4bit)이 돌려준 딕셔너리입니다."
assert list(model_metrics) == ["4비트 Base"], f"model_metrics의 키가 {list(model_metrics)}입니다. 지금은 \"4비트 Base\" 하나입니다."
recomputed = table_metrics(base_tables)
for name in METRIC_NAMES:
    assert abs(model_metrics["4비트 Base"][name] - recomputed[name]) < 1e-9, f"model_metrics[\"4비트 Base\"]의 {name}이 table_metrics(base_tables)와 다릅니다."
assert math.isfinite(model_metrics["4비트 Base"]["general_ppl"]) and model_metrics["4비트 Base"]["general_ppl"] > 1, "general_ppl은 1보다 큰 유한한 값입니다."
assert list(forgetting_table.index) == ["4비트 Base"] and list(forgetting_table.columns) == list(METRIC_NAMES.values()), "forgetting_table은 model_metrics의 모델이 행, 지표가 열이고 열 이름이 METRIC_NAMES의 테이블 이름인 테이블입니다."
print("✅ 통과!")

### LoRA 설정(제공 코드)

아래 셀은 과제 01과 같은 `lora_config`입니다. 실행만 하고 넘어갑니다. 3번에서 `SFTTrainer`에 넘기면 학습기가 4비트 Base에 어댑터를 붙입니다.

In [ ]:
from peft import LoraConfig

# 지난 시간 본실습과 같은 설정: 출력층을 뺀 모든 선형 층에 r=8 어댑터, 배율 16 / 8 = 2.0
lora_config = LoraConfig(
    r=8,                          # B, A 사이의 크기(rank)
    lora_alpha=16,                # 배율 = lora_alpha / r = 2.0
    target_modules="all-linear",  # lm_head를 뺀 모든 선형 층
    lora_dropout=0.05,            # 어댑터 입력에 거는 드롭아웃 비율
    task_type="CAUSAL_LM",        # 다음 토큰을 예측하는 생성 모델
)

## 3. QLoRA 학습: SFTConfig, SFTTrainer

**목적:** 4비트 Base에 어댑터를 붙여 쇼핑 상담 데이터로 학습하고 에폭마다 저장합니다.

**요구사항**
- (1) `SFTConfig`에 아래 값을 키워드 인자로 넣어 **`sft_config`** 를 만듭니다. `max_steps`는 넣지 않습니다.

```text
output_dir                   str(output_dir / "commerce_qlora_lr2e-4")
learning_rate                2e-4
num_train_epochs             4
per_device_train_batch_size  4
seed                         0
gradient_checkpointing       False
max_length                   256
eval_strategy                "epoch"
save_strategy                "epoch"
save_only_model              True
logging_steps                10
eval_on_start                True
report_to                    ["mlflow"]
run_name                     "commerce_qlora_lr_2e-4"
```

- (2) `SFTTrainer`에 `model`=`base_4bit`, `args`=`sft_config`, `train_dataset`=`train_dataset`, `eval_dataset`=`val_dataset`, `processing_class`=`tokenizer`, `peft_config`=`lora_config`를 넘긴 **`trainer`** 를 만듭니다. 4비트 모델과 `peft_config`를 받은 `SFTTrainer`는 어댑터를 붙여 베이스를 고정하고, 학습할 어댑터를 bf16으로 바꿉니다.
- (3) 학습기가 어댑터를 붙인 모델 `trainer.model`(`PeftModel`)을 **`qlora_model`** 에 담습니다.
- (4) `trainer.train()`의 반환값을 **`train_output`** 에 담습니다.
- (5) 학습 직후 `torch.cuda.max_memory_allocated()`(바이트)를 `1024 ** 3`으로 나눈 **`max_allocated_gb`**(GB)를 만듭니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 02 2-3 학습 설정과 학습 셀과 같은 방법입니다.
```

</details>

In [ ]:
# [작성] 학습 설정과 학습기를 만들고, 학습하며 최대 할당 메모리를 남기세요.
from trl import SFTConfig, SFTTrainer

# TODO (1) 지문의 값 열네 개를 한 줄에 하나씩 키워드 인자로 넣습니다(output_dir 는 문자열, max_steps 는 넣지 않습니다)
sft_config = SFTConfig(
    ...
)

# TODO (2) model, args, train_dataset, eval_dataset, processing_class, peft_config 인자를 한 줄에 하나씩 넣습니다(model 은 어댑터를 붙이지 않은 base_4bit)
trainer = SFTTrainer(
    ...
)

# TODO (3) 학습기가 어댑터를 붙인 모델을 담습니다
qlora_model = ...
qlora_model.print_trainable_parameters()

# MLflow run 하나에 설정, 학습 기록, 최대 할당 메모리를 남깁니다(run 을 여는 줄과 기록하는 줄은 그대로 씁니다)
with mlflow.start_run(run_name=sft_config.run_name):
    mlflow.log_params({"train_rows": len(train_dataset), "lora_r": 8, "load_in_4bit": True})

    # 학습 직전에 최대 메모리 기록을 지웁니다(그대로 씁니다)
    torch.cuda.reset_peak_memory_stats()

    # 학습을 시작하면 토크나이저의 PAD/BOS/EOS 토큰에 맞춰 모델 설정을 고쳤다는 안내가 나옵니다. 오류가 아닙니다
    # TODO (4) trainer 로 학습한 반환값을 담습니다
    train_output = ...

    # TODO (5) 학습 최대 할당 메모리를 GB 로 바꿔 담습니다
    max_allocated_gb = ...
    mlflow.log_metric("max_memory_allocated_gb", max_allocated_gb)
print("갱신 횟수:", train_output.global_step, "| 학습 최대 할당 메모리(GB):", round(max_allocated_gb, 2))

In [ ]:
# [자가채점] 설정 값, 학습기와 어댑터를 붙인 모델, 끝까지 돈 학습, 저장한 체크포인트, 최대 할당 메모리, 학습 뒤 어댑터를 확인합니다
from peft import PeftModel

assert isinstance(sft_config, SFTConfig), "sft_config는 trl의 SFTConfig입니다. `...`를 채웠는지 확인하세요."
assert abs(sft_config.learning_rate - 2e-4) < 1e-12, f"learning_rate가 {sft_config.learning_rate}입니다. 지문 테이블의 값을 넣으세요."
assert sft_config.num_train_epochs == 4 and sft_config.max_steps == -1 and sft_config.per_device_train_batch_size == 4, "num_train_epochs, per_device_train_batch_size 를 지문 테이블대로 넣고 max_steps는 넣지 마세요."
assert sft_config.max_length == 256 and sft_config.seed == 0 and sft_config.gradient_checkpointing is False, "max_length, seed, gradient_checkpointing 을 지문 테이블대로 넣으세요."
assert sft_config.eval_strategy == "epoch" and sft_config.save_strategy == "epoch" and sft_config.save_only_model is True, "eval_strategy, save_strategy, save_only_model 을 지문 테이블대로 넣으세요(에폭마다 어댑터를 저장합니다)."
assert sft_config.logging_steps == 10 and sft_config.eval_on_start is True, "logging_steps, eval_on_start 를 지문 테이블대로 넣으세요."
assert "mlflow" in sft_config.report_to and sft_config.run_name == "commerce_qlora_lr_2e-4", "report_to, run_name 을 지문 테이블대로 넣으세요."
assert sft_config.output_dir.endswith("commerce_qlora_lr2e-4"), f"output_dir가 {sft_config.output_dir}입니다. output 폴더 아래 commerce_qlora_lr2e-4 폴더의 경로를 문자열로 넣으세요."
assert isinstance(trainer, SFTTrainer) and trainer.args is sft_config, "trainer는 sft_config를 args로 넘긴 SFTTrainer입니다."
assert isinstance(qlora_model, PeftModel) and qlora_model is trainer.model, "qlora_model은 학습기가 어댑터를 붙인 trainer.model 입니다. SFTTrainer 에 peft_config=lora_config 를 넘겼는지 확인하세요."
trained_count = qlora_model.get_nb_trainable_parameters()[0]
assert trained_count == 5411328, f"학습 파라미터 수가 {trained_count:,}입니다(지난 과제 LoRA 5,411,328). model 에는 어댑터를 붙이지 않은 base_4bit 를, peft_config 에는 lora_config 를 넘겼는지 확인하세요."
assert trainer.eval_dataset is not None and len(trainer.eval_dataset) == len(val_dataset), "trainer의 검증 데이터 행 수가 val_dataset과 다릅니다. 검증 데이터로 val_dataset을 넘겼는지 확인하세요."
assert train_output.global_step == 136, f"global_step이 {train_output.global_step}입니다(기대값 136). trainer.train()이 끝까지 돌았는지 확인하세요."
for step in [34, 68, 102, 136]:
    assert (output_dir / "commerce_qlora_lr2e-4" / f"checkpoint-{step}" / "adapter_model.safetensors").exists(), f"checkpoint-{step} 에 어댑터가 없습니다. save_strategy 를 확인하세요."
model_gb = qlora_model.get_memory_footprint() / 1024 ** 3
gpu_total_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
assert model_gb < max_allocated_gb < gpu_total_gb, f"max_allocated_gb가 {max_allocated_gb}입니다. 셀에 적어 둔 기록 지우기 줄을 지우지 않았는지, 최대 할당 메모리를 1024 ** 3 으로 나눴는지 확인하세요."
trained_gate = qlora_model.base_model.model.model.layers[0].mlp.gate_proj
assert trained_gate.lora_A["default"].weight.dtype == torch.bfloat16, f"학습 뒤 A의 자료형이 {trained_gate.lora_A['default'].weight.dtype}입니다(기대값 bfloat16). SFTTrainer 는 4비트 모델에 붙인 어댑터를 bf16 으로 바꿉니다. model 에 4비트 Base(base_4bit)를, peft_config 에 lora_config 를 넘겼는지 확인하세요."
assert trained_gate.lora_B["default"].weight.abs().max().item() > 0, "학습 뒤 B가 아직 0입니다. 학습이 어댑터를 갱신했는지 확인하세요."
print("✅ 통과!")

## 4. 학습 후 답: eval, generate_replies

**목적:** 학습한 상담 모델의 답을 학습 전 답과 나란히 놓아 배포 전에 점검합니다.

**요구사항**
- (1) `qlora_model`을 평가 모드로 바꿔 어댑터의 드롭아웃을 끕니다.
- (2) `generate_replies`로 `qlora_model`이 1번과 같은 `check_questions`, `SYSTEM_PROMPT_COMMERCE`로 한 답의 리스트 **`after_answers`** 를 만듭니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 02 2-5 학습 전·후 답 셀과 같은 방법입니다.
교안 02 2-5 는 generate_replies 를 감싼 answer_check_questions 를 씁니다. generate_replies(model, questions, system_prompt) 는 교안 02 1-2 에 정의되어 있습니다.
```

</details>

In [ ]:
# [작성] 평가 모드로 바꾸고 같은 질문의 QLoRA 답을 테이블에 열로 더하세요.

# TODO (1) qlora_model 의 평가 모드 메서드를 부릅니다
...

# TODO (2) generate_replies 에 qlora_model 과 1번의 질문, 시스템 프롬프트를 넘깁니다
after_answers = ...

# 답 리스트를 1번 테이블의 "QLoRA 답변" 열로 더합니다(그대로 씁니다)
answer_table["QLoRA 답변"] = after_answers
display(answer_table)

In [ ]:
# [자가채점] 평가 모드와 새 열을 확인합니다
assert qlora_model.training is False, "답을 만들기 전에 qlora_model을 평가 모드로 바꾸세요(드롭아웃을 끕니다)."
assert list(answer_table.columns) == ["질문", "정답 답변", "4비트 Base 답변", "QLoRA 답변"], f"answer_table의 열이 {list(answer_table.columns)}입니다. 1번 테이블에 \"QLoRA 답변\" 열 하나를 더하세요."
assert list(answer_table["QLoRA 답변"]) == after_answers, "\"QLoRA 답변\" 열은 after_answers 입니다."
for answer in after_answers:
    assert isinstance(answer, str) and answer.strip() != "", "QLoRA 답변이 비어 있습니다. generate_replies 의 반환값을 담았는지 확인하세요."
assert after_answers != before_answers, "QLoRA 답변이 4비트 Base 답변과 네 개 모두 같습니다. 학습한 qlora_model에 다시 물었는지 확인하세요."
print("✅ 통과!")

### 에폭별 평가 준비(제공 코드)

아래 셀은 학습한 모델을 지우고, 에폭별 어댑터를 얹을 4비트 Base를 어댑터 없이 새로 불러옵니다(2번 지표와 같은 모델). 학습기가 `base_4bit` 안의 층에 어댑터를 붙였기 때문입니다. 실행만 하고 넘어갑니다.

In [ ]:
# 한 번만 실행합니다(다시 실행하면 지운 이름이 없어 NameError). 학습한 모델과 학습기를 지워 GPU 메모리를 비웁니다
import gc

del trainer, qlora_model, base_4bit
gc.collect()
torch.cuda.empty_cache()

# 2번과 같은 4비트 Base(어댑터 없음)
base_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,                      # 허브의 모델 이름
    quantization_config=bnb_config,  # 선형 층을 NF4 4비트로 불러오는 설정
    dtype=torch.bfloat16,            # 4비트로 바뀌지 않는 층(임베딩, 정규화)의 dtype
    device_map={"": 0},              # 모델 전체("")를 GPU 0번(첫 번째 GPU)에 둡니다
)

## 5. 에폭별 QLoRA 어댑터 평가: load_adapter, set_adapter

**목적:** 에폭마다 저장한 어댑터 4개를 같은 지표로 평가해, 어느 에폭을 쓸지 고를 근거를 만듭니다.

**요구사항**
- (1) 3번이 어댑터를 저장한 폴더 `output_dir / "commerce_qlora_lr2e-4"`를 **`qlora_dir`** 에 담습니다.
- (2) `qlora_dir` 안 `checkpoint-*` 폴더 이름의 `-` 뒤 갱신 횟수를 정수로 바꿔 오름차순 정렬한 리스트 `steps`를 만듭니다.
- (3) 에폭(1부터)마다 키가 `"이름"`(`f"ep{에폭}"`, 예: `ep1`), `"에폭"`, `"갱신 횟수"`, `"폴더"`(체크포인트 폴더의 `Path` 객체)이고 이 순서인 딕셔너리를 리스트 **`checkpoint_rows`** 에 담습니다.
- (4) 지난 과제와 같이 `PeftModel.from_pretrained`로 `base_4bit`에 첫 행의 어댑터를 `adapter_name`을 그 행의 `"이름"`으로 얹은 **`qlora_model`**(`PeftModel`)을 만듭니다.
- (5) 나머지 행의 어댑터를 `qlora_model.load_adapter`로 그 행의 `"이름"`을 `adapter_name`으로 붙여 더 얹습니다.
- (6) `checkpoint_rows` 순서대로 `qlora_model.set_adapter`로 그 행의 어댑터를 `inference_mode=True`(켠 어댑터를 학습하지 않게 고정)로 켭니다.
- (7) 켠 모델을 `evaluate_model`로 평가한 반환값을 딕셔너리 **`epoch_tables`**(키는 `"이름"`)에 담습니다.
- (8) `table_metrics`로 계산한 지표를 딕셔너리 **`epoch_metrics`**(키는 `"이름"`)와 2번의 **`model_metrics`**(같은 키, `"4비트 Base"` 뒤)에 담습니다.
- (9) 어댑터 지표까지 더한 `model_metrics`로 2번과 같이 모델이 행, 열 이름이 `METRIC_NAMES`의 테이블 이름인 **`forgetting_table`** 을 다시 만듭니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 02 3-2 에폭별 어댑터 4개 얹기, 3-3 에폭별 어댑터 평가 셀과 같은 방법입니다.
```

</details>

In [ ]:
# [작성] 체크포인트 리스트를 만들고, 4비트 Base 하나에 어댑터 4개를 얹으세요.
from peft import PeftModel

# TODO (1) output_dir 아래 3번의 학습 폴더를 Path 로 담습니다
qlora_dir = ...

# TODO (2) qlora_dir.glob("checkpoint-*") 로 폴더를 돌며 이름의 "-" 뒤를 int 로 바꿔 steps 에 더하고 정렬합니다
steps = []
...

# TODO (3) enumerate(steps, start=1) 로 에폭마다 "이름", "에폭", "갱신 횟수", "폴더" 딕셔너리를 더합니다
checkpoint_rows = []
...

# TODO (4) PeftModel.from_pretrained 에 base_4bit, 첫 행의 "폴더", adapter_name 을 넘깁니다
first_row = checkpoint_rows[0]
qlora_model = ...

# TODO (5) 나머지 행은 qlora_model.load_adapter 로 "폴더" 와 adapter_name 을 넘겨 더 얹습니다
for row in checkpoint_rows[1:]:
    ...
display(pd.DataFrame(checkpoint_rows))

In [ ]:
# [자가채점] 체크포인트 리스트와 얹은 어댑터를 확인합니다
expected_names = ['ep1', 'ep2', 'ep3', 'ep4']
assert qlora_dir == output_dir / "commerce_qlora_lr2e-4", "qlora_dir는 output 폴더 아래 commerce_qlora_lr2e-4 폴더입니다."
assert isinstance(checkpoint_rows, list) and len(checkpoint_rows) == 4, f"checkpoint_rows가 {len(checkpoint_rows)}행입니다(기대값 4). qlora_dir 에는 이번 학습이 저장한 checkpoint-<갱신 횟수> 폴더 4개만 있어야 합니다. 이전 실행의 체크포인트가 남아 있으면 그 폴더를 지우고 노트북을 처음부터 다시 실행해 학습하세요."
names = []
steps_by_row = []
for row in checkpoint_rows:
    assert list(row) == ["이름", "에폭", "갱신 횟수", "폴더"], f"행의 키가 {list(row)}입니다. 지문의 키 네 개를 순서대로 담으세요."
    assert isinstance(row["폴더"], Path), f"{row['이름']} 행의 \"폴더\"가 Path 객체가 아닙니다. qlora_dir 에 폴더 이름을 이어 붙인 Path 객체로 담으세요."
    names.append(row["이름"])
    steps_by_row.append(row["갱신 횟수"])
assert names == expected_names, f"이름이 {names}입니다. 에폭 순서로 ep1부터 담으세요."
assert steps_by_row == [34, 68, 102, 136], f"갱신 횟수가 {steps_by_row}입니다(기대값 [34, 68, 102, 136]). 폴더 이름의 갱신 횟수를 정수로 바꿔 정렬했는지 확인하세요. 다른 실행의 체크포인트가 섞였으면 qlora_dir 를 지우고 노트북을 처음부터 다시 실행해 학습하세요."
assert isinstance(qlora_model, PeftModel), "qlora_model은 첫 행의 어댑터를 base_4bit에 얹어 만든 PeftModel입니다."
assert sorted(qlora_model.peft_config.keys()) == sorted(expected_names), f"얹은 어댑터가 {list(qlora_model.peft_config.keys())}입니다. 나머지 행의 어댑터도 이름을 붙여 더 얹으세요."
print("✅ 통과!")

In [ ]:
# [작성] 어댑터를 하나씩 켜서 평가하고, 지표 테이블을 다시 만드세요.
epoch_tables = {}
epoch_metrics = {}
for row in checkpoint_rows:
    # TODO (6) qlora_model.set_adapter 로 row["이름"] 어댑터를 inference_mode 로 켭니다
    ...

    # 판사 호출마다 나오는 DeepEval 요약과 경고는 2번과 같으며 오류가 아닙니다
    # TODO (7) evaluate_model 로 평가해 epoch_tables[row["이름"]] 에 담습니다
    ...

    # TODO (8) table_metrics 로 계산한 지표를 epoch_metrics 와 model_metrics 의 같은 이름 키에 담습니다
    ...

# TODO (9) 2번과 같이 model_metrics 로 테이블을 만듭니다
forgetting_table = ...
display(forgetting_table.round(3))

In [ ]:
# [자가채점] 어댑터별 채점 딕셔너리와 지표, 테이블의 행 순서, 켜 둔 어댑터를 확인합니다
expected_names = ['ep1', 'ep2', 'ep3', 'ep4']
assert list(epoch_tables) == expected_names and list(epoch_metrics) == expected_names, "epoch_tables, epoch_metrics의 키는 checkpoint_rows의 이름 순서입니다."
for name in expected_names:
    assert list(epoch_tables[name]) == ["train", "domain", "refusal", "general", "ppl"], f"epoch_tables[{name}]은 evaluate_model이 돌려준 딕셔너리입니다."
    recomputed = table_metrics(epoch_tables[name])
    for metric_name in METRIC_NAMES:
        assert abs(epoch_metrics[name][metric_name] - recomputed[metric_name]) < 1e-9, f"epoch_metrics[{name}]의 {metric_name}이 table_metrics(epoch_tables[{name}])와 다릅니다."
    assert model_metrics[name] == epoch_metrics[name], f"model_metrics[{name}]에 epoch_metrics[{name}]을 담으세요."
assert list(model_metrics) == ["4비트 Base"] + expected_names, f"model_metrics의 키가 {list(model_metrics)}입니다. 4비트 Base 뒤에 어댑터 이름을 차례로 더하세요."
assert list(forgetting_table.index) == ["4비트 Base"] + expected_names, "forgetting_table을 어댑터 지표까지 더한 model_metrics로 다시 만드세요."
assert qlora_model.active_adapter == expected_names[-1], f"켜진 어댑터가 {qlora_model.active_adapter}입니다. 반복 안에서 평가하기 전에 그 행의 어댑터를 켜세요."
trainable_names = []
for param_name, param in qlora_model.named_parameters():
    if param.requires_grad:
        trainable_names.append(param_name)
assert trainable_names == [], f"켠 어댑터의 파라미터 {len(trainable_names)}개가 학습 가능(requires_grad=True)으로 남았습니다. set_adapter 에 inference_mode=True 를 넘겨 켠 어댑터를 고정하세요."
domain_answer_sets = set()
for name in expected_names:
    domain_answer_sets.add(tuple(epoch_tables[name]["domain"]["답변"]))
assert len(domain_answer_sets) > 1, "어댑터 4개의 도메인 답이 모두 같습니다. 반복 안에서 어댑터를 바꿔 켠 뒤 evaluate_model을 불렀는지 확인하세요."
print("✅ 통과!")

## QLoRA Tuned 고르기(제공 코드)

지난 과제 02와 같은 규칙으로 고릅니다. 학습 데이터 정답률 0.8 이상인 후보 가운데 쇼핑 상담 평가셋 통과 수가 가장 많은 것, 후보가 없으면 학습 데이터 정답률이 가장 높은 것입니다(같으면 앞 에폭). 실행만 하고 넘어갑니다.

- **결과:** `tuned_row`, `tuned_name`(뒤의 6번, 7번에 씁니다)

In [ ]:
TRAIN_ACC_MIN = 0.8  # 규칙 1: 학습 데이터 정답률이 이 값 이상인 어댑터만 후보

# 규칙 2: 후보 가운데 쇼핑 상담 평가셋 통과 수가 가장 많은 것(앞 에폭부터 보므로 같으면 앞 에폭)
tuned_row = None
for row in checkpoint_rows:
    passes = epoch_tables[row["이름"]]["domain"]["통과"].sum()  # 재현 + 적용 통과 문항 수
    if epoch_metrics[row["이름"]]["train_acc"] >= TRAIN_ACC_MIN:
        if tuned_row is None or passes > best_passes:
            tuned_row = row
            best_passes = passes

# 규칙 3: 후보가 없으면 학습 데이터 정답률이 가장 높은 것(같으면 앞 에폭)
if tuned_row is None:
    tuned_row = checkpoint_rows[0]
    for row in checkpoint_rows:
        if epoch_metrics[row["이름"]]["train_acc"] > epoch_metrics[tuned_row["이름"]]["train_acc"]:
            tuned_row = row

# 고른 어댑터의 이름
tuned_name = tuned_row["이름"]
print("QLoRA Tuned:", tuned_name, "| 에폭:", tuned_row["에폭"], "| 폴더:", tuned_row["폴더"])

## 6. 지난 과제 LoRA Tuned와 비교: DataFrame, astype

**목적:** 4비트 위에서 학습한 QLoRA가 지난 과제의 float32 LoRA만큼 잘하는지 한 테이블로 비교합니다.

**요구사항**
- (1) (선택) 지난 과제 02 3번 `full_compare_table`의 두 번째 행(행 이름이 `LoRA Tuned <에폭>에폭(step <갱신 횟수>)` 모양인 행) 값을 같은 지표 키에 옮겨 적습니다. 값이 없으면 `None`으로 둡니다.
- (2) 행이 `"4비트 Base"`(`model_metrics["4비트 Base"]`), `"QLoRA Tuned"`(`epoch_metrics[tuned_name]`), `"LoRA Tuned(지난 과제)"`(`lora_tuned_metrics`) 순서이고 열 이름이 `METRIC_NAMES`의 테이블 이름인 `pd.DataFrame` **`qlora_compare_table`** 을 만듭니다.
- (3) `qlora_compare_table`의 모든 열을 소수(`float64`, `None`은 `NaN`)로 바꿉니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 02 4절 LoRA·풀 파인튜닝과 비교 셀과 같은 방법입니다. 행 이름을 키로, 지표 딕셔너리를 값으로 둔 딕셔너리로 테이블을 만들고 뒤집습니다.
```

</details>

In [ ]:
# [작성] (선택) 지난 과제 LoRA Tuned 지표를 옮겨 적고, 비교 테이블을 만드세요.

# TODO (1) (선택) 지난 과제 full_compare_table 의 "LoRA Tuned <에폭>에폭(step <갱신 횟수>)" 행을 옮겨 적습니다
# 그 테이블의 열 이름은 METRIC_NAMES 의 값이므로 키와 짝지어 숫자를 적습니다(비율은 0~1, general_ppl 은 1보다 큰 값 그대로)
lora_tuned_metrics = {
    "train_acc": None,
    "domain_recall_acc": None,
    "domain_apply_acc": None,
    "over_refusal_rate": None,
    "refusal_rate": None,
    "general_acc": None,
    "general_ppl": None,
}

# TODO (2) 세 행 이름을 키, 지표 딕셔너리를 값으로 묶어 2번과 같이 행과 열을 바꾸고 열 이름을 바꿉니다
# TODO (3) astype 으로 모든 열을 float 로 바꿉니다
qlora_compare_table = ...
display(qlora_compare_table.round(3))

In [ ]:
# [자가채점] 옮겨 적은 지난 과제 지표와 비교 테이블을 확인합니다
assert isinstance(lora_tuned_metrics, dict) and list(lora_tuned_metrics) == list(METRIC_NAMES), "lora_tuned_metrics의 키는 METRIC_NAMES의 키 일곱 개를 같은 순서로 둡니다."
for name in METRIC_NAMES:
    value = lora_tuned_metrics[name]
    if value is not None:
        assert isinstance(value, (int, float)), f"lora_tuned_metrics[\"{name}\"]에 숫자를 옮겨 적거나 None 그대로 두세요."
        if name == "general_ppl":
            assert value > 1, "general_ppl은 1보다 큰 값입니다."
        else:
            assert 0 <= value <= 1, f"lora_tuned_metrics[\"{name}\"]는 0~1 사이 비율입니다."
assert list(qlora_compare_table.index) == ['4비트 Base', 'QLoRA Tuned', 'LoRA Tuned(지난 과제)'], f"qlora_compare_table의 행이 {list(qlora_compare_table.index)}입니다. 지문의 행 이름과 순서를 확인하세요."
assert list(qlora_compare_table.columns) == list(METRIC_NAMES.values()), "qlora_compare_table의 열은 METRIC_NAMES의 테이블 이름입니다."
for column in qlora_compare_table.columns:
    assert str(qlora_compare_table[column].dtype) == "float64", f"{column} 열이 {qlora_compare_table[column].dtype}입니다. None 이 섞여도 소수 테이블이 되게 바꾸세요."
for name, column in METRIC_NAMES.items():
    assert abs(qlora_compare_table.loc["QLoRA Tuned", column] - epoch_metrics[tuned_name][name]) < 1e-9, "QLoRA Tuned 행은 epoch_metrics[tuned_name]입니다."
    assert abs(qlora_compare_table.loc["4비트 Base", column] - model_metrics["4비트 Base"][name]) < 1e-9, "4비트 Base 행은 model_metrics[\"4비트 Base\"]입니다."
print("✅ 통과!")

## 7. QLoRA 어댑터 병합: PeftModel.from_pretrained, merge_and_unload

**목적:** 고른 어댑터를 원래 정밀도 베이스에 합쳐 어댑터 없이 배포할 모델을 만들고, 병합 모델로 답을 얻습니다.

**요구사항**
- (1) `MODEL_NAME`을 `torch.float32`로 새로 불러 `device`에 올린 **`merge_base`** 를 만듭니다(`base_4bit`에는 병합하지 않습니다).
- (2) QLoRA Tuned 어댑터 폴더 `tuned_row["폴더"]`를 **`tuned_dir`** 에 담고, `PeftModel.from_pretrained`로 `merge_base`에 `tuned_dir`의 어댑터 하나만 얹고 `merge_and_unload()`로 병합한 **`merged_model`** 을 만듭니다.
- (3) `generate_replies`로 `merged_model`이 `SYSTEM_PROMPT_COMMERCE`로 1번의 `check_questions`에 한 답의 리스트 **`merged_answers`** 를 만듭니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 02 5-2 원래 정밀도 베이스에 병합 셀과 같은 방법입니다.
답을 얻는 방법은 1번, 4번의 generate_replies 와 같습니다.
```

</details>

In [ ]:
# [작성] float32 베이스에 QLoRA Tuned 어댑터를 병합하고, 병합 모델로 1번 질문에 답하세요.

# TODO (1) AutoModelForCausalLM.from_pretrained 로 float32 베이스를 새로 불러와 device 에 올립니다
merge_base = ...

# 병합 전 첫 층 gate_proj 가중치 일부를 복사해 둡니다. 병합하면 이 값에 어댑터가 더해집니다(그대로 씁니다)
before_weight = merge_base.model.layers[0].mlp.gate_proj.weight[0, :8].clone()

# TODO (2) tuned_row 의 "폴더" 를 tuned_dir 에 담고, PeftModel.from_pretrained 에 merge_base 와 tuned_dir 를 넘긴 뒤 merge_and_unload 를 부릅니다
tuned_dir = ...
merged_model = ...

# TODO (3) generate_replies 에 merged_model, check_questions, SYSTEM_PROMPT_COMMERCE 를 넘깁니다
merged_answers = ...

# 답 리스트를 1번 테이블의 "병합 모델 답변" 열로 더합니다(그대로 씁니다)
answer_table["병합 모델 답변"] = merged_answers
display(answer_table)

In [ ]:
# [자가채점] 병합 모델의 구조와 병합 모델의 답을 확인합니다
assert next(merge_base.parameters()).dtype == torch.float32, "merge_base는 MODEL_NAME을 float32로 새로 불러온 베이스입니다. 4비트 Base(base_4bit)에 병합하지 마세요."
assert next(merged_model.parameters()).device.type == device.type, f"merged_model이 {next(merged_model.parameters()).device}에 있습니다. merge_base를 불러온 뒤 .to(device)로 {device}에 올리세요."
assert tuned_dir == tuned_row["폴더"], f"tuned_dir가 {tuned_dir}입니다. 제공 코드가 고른 QLoRA Tuned 행의 폴더 tuned_row[\"폴더\"]를 담고 그 어댑터를 병합하세요."
assert not isinstance(merged_model, PeftModel), "merged_model이 아직 PeftModel입니다. 어댑터를 얹은 모델을 병합한 반환값을 담으세요."
merged_gate = merged_model.model.layers[0].mlp.gate_proj
assert type(merged_gate) is torch.nn.Linear and merged_gate.weight.dtype == torch.float32, f"merged_model 첫 층 gate_proj가 {type(merged_gate).__name__}({merged_gate.weight.dtype})입니다. float32로 새로 불러온 merge_base에 어댑터를 얹어 병합하세요."
assert not torch.equal(merged_gate.weight[0, :8], before_weight), "병합한 첫 층 gate_proj 가중치가 병합 전 before_weight 와 같습니다. merge_base 에 tuned_dir 의 어댑터를 얹고 merge_and_unload 로 병합한 반환값을 담으세요."
assert isinstance(merged_answers, list) and len(merged_answers) == len(check_questions), f"merged_answers는 check_questions {len(check_questions)}개에 대한 답 리스트입니다."
for answer in merged_answers:
    assert isinstance(answer, str) and answer.strip() != "", "병합 모델 답변이 비어 있습니다. generate_replies 의 반환값을 담았는지 확인하세요."
assert merged_answers != before_answers, "병합 모델 답변이 학습 전 4비트 Base 답변과 네 개 모두 같습니다. 병합한 merged_model 에 물었는지 확인하세요."
assert list(answer_table["병합 모델 답변"]) == merged_answers, "\"병합 모델 답변\" 열은 merged_answers 입니다."
print("✅ 통과!")